# CLTV Research MVP

Стартовый notebook для первого end-to-end прототипа research assistant по теме "Применение CLTV в иностранных банках".

Цель текущей версии: пройти путь от темы к первым evidence-кандидатам: research plan, curated seed sources, fetching/parsing, chunking, baseline filtering, BM25 ranking и evidence table.

## Как читать этот notebook

Notebook оформлен как прозрачный research pipeline. Каждый блок отвечает на два вопроса:

1. Что делает метод технически.
2. Зачем этот метод нужен в банковском research assistant.

Важно: этот MVP не пытается сразу быть production-системой. Мы сначала делаем воспроизводимый контур на seed-источниках, чтобы потом безопасно заменить отдельные части: web search, HTML parser, PDF parser, BM25, embedding-reranker или LLM.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROJECT_ROOT

## 1. Ввод темы

Для первого демо используем фиксированную тему из проектных материалов. Позже это станет пользовательским input в Streamlit или FastAPI UI.

Почему начинаем с одной темы:

- легче проверить качество источников вручную;
- проще объяснить результат на защите;
- можно измерить baseline-метрики: Precision@K, noise reduction, citation accuracy;
- один стабильный сценарий помогает отладить pipeline до подключения внешнего поиска.

In [ ]:
topic = "CLTV in foreign banks"
topic

## 2. Research Planner

Research Planner превращает широкую тему в набор блоков исследования и поисковых запросов.

В production-версии planner может использовать LLM и возвращать JSON. В первом MVP используем rule-based planner:

- он не зависит от API-ключей;
- работает одинаково при каждом запуске;
- задает понятную структуру будущего отчета;
- снижает риск того, что LLM уведет исследование в сторону.

In [ ]:
from research_assistant.planner import build_cltv_research_plan

plan = build_cltv_research_plan(topic)
print("Research blocks:")
for block in plan.blocks:
    print(f"- {block}")

print("\nSearch queries:")
for query in plan.queries:
    print(f"- [{query.research_block}] {query.query}")

## 3. Seed Source Collector

Пока внешний web search не реализован, используем curated CSV. Это делает демо воспроизводимым: pipeline не зависит от API-ключей, лимитов поисковиков и случайных изменений выдачи.

Метод:

1. Команда заранее собирает надежные публичные URL.
2. Каждому источнику назначается `source_id`.
3. Источник привязывается к research block.
4. Collector валидирует структуру CSV и URL.
5. Дальше весь pipeline работает не с "просто ссылками", а с typed metadata.

Это важно для банка: у каждого будущего тезиса должен быть источник, тип источника и traceability.

In [ ]:
from research_assistant.collector import group_sources_by_research_block, load_seed_sources

seed_path = PROJECT_ROOT / "data" / "seed_sources" / "cltv_sources_template.csv"
sources = load_seed_sources(seed_path)
sources_by_id = {source.source_id: source for source in sources}

print(f"Loaded ready seed sources: {len(sources)}")
for source in sources:
    print(f"- {source.source_id}: {source.publisher} | {source.title}")

## 4. Покрытие research blocks источниками

Проверяем, что seed-набор покрывает основные блоки исследования, а не состоит из источников одного типа.

Почему это важно:

- отчет по CLTV должен объяснять бизнес-ценность, методы расчета, данные, use cases и риски;
- если источники покрывают только marketing/use cases, отчет будет слабым по методологии;
- если источники покрывают только академические модели, отчет будет хуже для бизнес-команды.

In [ ]:
grouped_sources = group_sources_by_research_block(sources)

for block in plan.blocks:
    block_sources = grouped_sources.get(block, [])
    print(f"{block}: {len(block_sources)} source(s)")
    for source in block_sources:
        print(f"  - {source.source_id} [{source.source_type.value}] {source.publisher}: {source.title}")

## 5. Минимальная таблица источников

Эта таблица станет входом для следующего шага: загрузки HTML/PDF и извлечения clean text.

Поля:

- `source_id` - стабильный ID для ссылок в evidence table;
- `publisher` - помогает оценивать доверие к источнику;
- `source_type` - пригодится для trust score;
- `research_block` - показывает, какую часть исследования источник поддерживает;
- `url` - исходная ссылка для аудита.

In [ ]:
source_rows = [
    {
        "source_id": source.source_id,
        "publisher": source.publisher,
        "source_type": source.source_type.value,
        "research_block": source.research_block,
        "title": source.title,
        "url": str(source.url),
    }
    for source in sources
]

source_rows[:3]

## 6. Fetching: загрузка raw content

Fetching отвечает за скачивание исходного материала и сохранение его в `data/raw`.

Почему мы сохраняем raw:

- можно повторно парсить страницу без нового запроса в интернет;
- можно проверить, какой именно документ видел pipeline;
- можно сравнивать версии parser/cleaner на одинаковом входе;
- это основа для auditability.

В MVP используется простой sequential fetcher на `urllib`. В более зрелой версии его можно заменить на:

- `httpx` для асинхронной загрузки;
- `curl_cffi` для более стабильной загрузки отдельных публичных страниц;
- Playwright как fallback для JS-страниц;
- Search API/RSS/allowlist вместо ручного CSV.

По умолчанию live-fetch выключен, чтобы notebook можно было открыть без сети. Чтобы скачать первые источники, поставьте `RUN_LIVE_FETCH = True`.

Batch-режим сделан устойчивым к ошибкам: если один сайт отдает 403, timeout или недоступен, notebook продолжает работу с остальными источниками и показывает failure report.

In [ ]:
from research_assistant.fetcher import fetch_sources_safe, raw_document_path

for source in sources[:5]:
    print(f"{source.source_id} -> {raw_document_path(source, RAW_DIR).relative_to(PROJECT_ROOT)}")

In [ ]:
RUN_LIVE_FETCH = False
FETCH_LIMIT = 8

if RUN_LIVE_FETCH:
    fetch_results = fetch_sources_safe(sources, RAW_DIR, limit=FETCH_LIMIT, timeout_seconds=25)
    raw_documents = [result.raw_document for result in fetch_results if result.ok and result.raw_document]
    print(f"Fetch ok: {len(raw_documents)} / {len(fetch_results)}")
    for result in fetch_results:
        if result.ok and result.raw_document:
            status = "cache" if result.raw_document.from_cache else f"http {result.raw_document.status_code}"
            print(f"OK   {result.source_id}: {status} -> {result.raw_document.path.relative_to(PROJECT_ROOT)}")
        else:
            print(f"FAIL {result.source_id}: {result.error}")
else:
    fetch_results = []
    raw_documents = []
    print("Live fetch is disabled. Set RUN_LIVE_FETCH = True to download raw source files.")

## 7. Parsing: извлечение clean text

Parser превращает raw HTML/PDF в чистый текст для chunking и ранжирования.

Метод в MVP:

- HTML: базовый parser на стандартной библиотеке `html.parser`;
- удаляются `script`, `style`, `nav`, `header`, `footer` и другие шумные блоки;
- PDF: `pypdf`, если установлен;
- результат сохраняется в `data/clean/{source_id}.txt`.

Почему это отдельный шаг:

- загрузка и парсинг ломаются по разным причинам;
- HTML parser можно улучшать независимо от collector;
- clean text нужен для BM25, embeddings, evidence chunks и LLM synthesis;
- в банковском контуре важно понимать, какой parser создал конкретный текст.

In [ ]:
from research_assistant.parser import parse_raw_documents_safe

if raw_documents:
    parse_results = parse_raw_documents_safe(raw_documents, sources, CLEAN_DIR)
    clean_documents = [result.clean_document for result in parse_results if result.ok and result.clean_document]
    print(f"Parse ok: {len(clean_documents)} / {len(parse_results)}")
    for result in parse_results:
        if result.ok and result.clean_document:
            clean_document = result.clean_document
            print(
                f"OK   {clean_document.source_id}: {clean_document.char_count} chars | "
                f"{clean_document.parser_name} -> {clean_document.path.relative_to(PROJECT_ROOT)}"
            )
        else:
            print(f"FAIL {result.source_id}: {result.error}")
else:
    parse_results = []
    clean_documents = []
    print("No raw documents in this notebook run. Enable live fetch first or place files in data/raw.")

## 8. Загрузка cached clean documents

Если live-fetch уже запускался раньше, clean-файлы лежат в `data/clean`. Эта ячейка подхватывает их без повторного скачивания.

Это полезно для демо:

- можно один раз скачать источники и дальше работать офлайн;
- ranking и evidence можно отлаживать на стабильном тексте;
- notebook не зависит от состояния сайтов в момент презентации.

In [ ]:
from research_assistant.models import CleanDocument

if not clean_documents:
    cached_clean_documents = []
    for clean_path in sorted(CLEAN_DIR.glob("*.txt")):
        source_id = clean_path.stem
        source = sources_by_id.get(source_id)
        if source is None:
            continue
        text = clean_path.read_text(encoding="utf-8")
        cached_clean_documents.append(
            CleanDocument(
                source_id=source.source_id,
                title=source.title,
                url=source.url,
                path=clean_path,
                text=text,
                parser_name="cached_clean_text",
                char_count=len(text),
            )
        )
    clean_documents = cached_clean_documents

print(f"Clean documents available for chunking: {len(clean_documents)}")
for document in clean_documents:
    print(f"- {document.source_id}: {document.char_count} chars from {document.path.relative_to(PROJECT_ROOT)}")

## 9. Chunking: разбиваем документы на фрагменты

LLM и search/ranking не должны работать с огромным документом целиком. Вместо этого документ режется на chunks - небольшие фрагменты, которые можно отдельно оценивать и цитировать.

Почему chunking нужен:

- один документ может покрывать несколько разных research blocks;
- BM25 и embeddings лучше работают на коротких фрагментах;
- evidence table должна ссылаться на конкретный фрагмент, а не на всю страницу;
- потом LLM получает только top evidence, а не весь шумный текст.

Метод в MVP:

- chunk size около 1200 символов;
- небольшой overlap, чтобы не потерять контекст на границе;
- слишком короткие chunks отбрасываются.

In [ ]:
from research_assistant.chunker import chunk_clean_documents

chunks = chunk_clean_documents(
    clean_documents,
    sources,
    max_chars=1200,
    overlap_chars=150,
    min_chars=160,
)

print(f"Chunks produced: {len(chunks)}")
for chunk in chunks[:5]:
    print(f"- {chunk.chunk_id}: {chunk.char_count} chars | {chunk.research_block}")

## 10. Baseline noise filtering

Noise Filter удаляет фрагменты, которые почти наверняка не помогут отчету.

В MVP используем простые правила:

- убрать слишком короткие chunks;
- убрать очевидный служебный шум: cookies, newsletter, terms, advertisement, consent forms, country selectors;
- оставить только chunks с банковскими/CLTV-доменными терминами;
- убрать точные или почти точные дубли по начальным токенам.

Почему начинаем с эвристик:

- они прозрачны и легко объясняются;
- они быстрые;
- они дают baseline для сравнения с embeddings/LLM reranker;
- в банковском проекте прозрачность ранних фильтров полезна для аудита.

In [ ]:
from research_assistant.filtering import filter_chunks

filtered_chunks = filter_chunks(chunks, min_chars=200, min_domain_terms=2)

print(f"Chunks before filtering: {len(chunks)}")
print(f"Chunks after filtering:  {len(filtered_chunks)}")
if chunks:
    reduction = 1 - len(filtered_chunks) / len(chunks)
    print(f"Noise reduction estimate: {reduction:.1%}")

## 11. BM25 ranking

BM25 - классический lexical ranking алгоритм. Он сравнивает поисковый запрос с chunks по словам, но учитывает не только количество совпадений:

- редкие слова важнее частых;
- слишком длинные chunks не получают несправедливое преимущество;
- повтор одного и того же слова насыщается, а не растет бесконечно.

Почему BM25 хороший baseline:

- не требует GPU и LLM;
- работает быстро;
- легко объясняется;
- хорошо подходит для первого отбора evidence.

Где BM25 слабее следующих методов:

- плохо понимает синонимы;
- не ловит смысл без совпадения слов;
- хуже multilingual/semantic search;
- может пропускать полезные фрагменты, если формулировки отличаются.

Позже BM25 можно дополнить embeddings и LLM reranker.

In [ ]:
from research_assistant.filtering import rank_chunks_bm25

ranked_chunks = rank_chunks_bm25(filtered_chunks, plan.queries, top_k_per_query=3)

print(f"Ranked chunk-query matches: {len(ranked_chunks)}")
for chunk, query, score in ranked_chunks[:10]:
    print(f"{score:.3f} | {query.research_block} | {chunk.chunk_id} | {query.query}")

## 12. Evidence table

Evidence table - это мост между найденными источниками и будущим отчетом.

Каждый evidence item хранит:

- `source_id`;
- `chunk_id`;
- фрагмент текста;
- ссылку;
- тип источника;
- research block;
- запрос, по которому фрагмент был найден;
- relevance score.

Зачем это нужно:

- LLM Synthesizer будет писать отчет только по evidence;
- аналитик сможет проверить каждый тезис;
- Quality Gate сможет ловить утверждения без источников;
- команда сможет считать citation accuracy и hallucination rate.

In [ ]:
from research_assistant.evidence import build_evidence_items, write_evidence_csv, write_evidence_jsonl

evidence_items = build_evidence_items(ranked_chunks, max_items=20)

print(f"Evidence items: {len(evidence_items)}")
for item in evidence_items[:5]:
    print(f"#{item.rank} {item.source_id}/{item.chunk_id} score={item.relevance_score} query={item.matched_query}")
    print(item.text[:220].replace("\n", " ") + "...")
    print()

In [ ]:
if evidence_items:
    evidence_csv_path = write_evidence_csv(evidence_items, REPORTS_DIR / "evidence_cltv.csv")
    evidence_jsonl_path = write_evidence_jsonl(evidence_items, REPORTS_DIR / "evidence_cltv.jsonl")
    print(f"CSV:   {evidence_csv_path.relative_to(PROJECT_ROOT)}")
    print(f"JSONL: {evidence_jsonl_path.relative_to(PROJECT_ROOT)}")
else:
    print("No evidence items yet. Fetch/parse at least one source, then rerun chunking and ranking.")

## 13. Evaluation summary

Evaluation summary фиксирует, что реально получилось в запуске notebook.

Это не финальная бизнес-оценка качества, а технический контроль MVP:

- сколько seed-источников было в наборе;
- сколько документов успешно распарсено;
- сколько chunks было до и после фильтрации;
- какая доля шума удалена;
- какие research blocks покрыты evidence;
- какие типы источников есть в текущем наборе.

Зачем это нужно: без такого summary трудно понять, улучшился pipeline или просто сгенерировал текст. Для следующего этапа эти показатели станут baseline.

In [ ]:
from research_assistant.evaluation import build_evaluation_summary, write_evaluation_json

evaluation_summary = build_evaluation_summary(
    plan=plan,
    sources=sources,
    clean_documents=clean_documents,
    chunks=chunks,
    filtered_chunks=filtered_chunks,
    evidence_items=evidence_items,
)

evaluation_path = write_evaluation_json(evaluation_summary, REPORTS_DIR / "evaluation_cltv.json")

for key, value in evaluation_summary.items():
    print(f"{key}: {value}")
print(f"Saved: {evaluation_path.relative_to(PROJECT_ROOT)}")

## 14. Первый Markdown-отчет

На этом шаге мы не подключаем LLM. Вместо этого строим template-based отчет строго по evidence.

Почему так:

- сначала нужно убедиться, что evidence table достаточно качественная;
- template report не галлюцинирует новые факты;
- аналитик видит, какие источники реально поддерживают каждый раздел;
- LLM Synthesizer можно добавить позже как заменяемый слой поверх того же evidence.

Ограничение: такой отчет выглядит менее гладко, чем LLM-сводка, зато он прозрачен и проверяем.

In [ ]:
from research_assistant.report import render_markdown_report, write_markdown_report

report_markdown = render_markdown_report(
    topic=topic,
    evidence_items=evidence_items,
    evaluation_summary=evaluation_summary,
)
report_path = write_markdown_report(report_markdown, REPORTS_DIR / "report_cltv.md")

print(f"Saved: {report_path.relative_to(PROJECT_ROOT)}")
print("\n".join(report_markdown.splitlines()[:35]))

## 15. Quality Gate

Quality Gate проверяет, можно ли считать notebook output приемлемым для MVP.

Проверки в текущей версии:

- достаточно clean-документов;
- достаточно evidence-фрагментов;
- evidence идет из нескольких источников;
- есть coverage по ключевым research blocks;
- отчет содержит `Unknowns`;
- отчет содержит evidence table;
- у каждого evidence item есть `source_id` и `chunk_id`.

Статусы:

- `pass` - все обязательные и желательные проверки прошли;
- `warn` - обязательные проверки прошли, но есть слабые места;
- `fail` - результат нельзя принимать как MVP output.

In [ ]:
from research_assistant.quality_gate import run_quality_gate

quality_gate_result = run_quality_gate(
    evidence_items=evidence_items,
    evaluation_summary=evaluation_summary,
    report_markdown=report_markdown,
)

print(f"Quality gate status: {quality_gate_result.status}")
for check in quality_gate_result.checks:
    marker = "PASS" if check.passed else check.severity.upper()
    print(f"{marker:>4} | {check.name}: {check.detail}")

## 16. Закрытие Этапа 1

Этап 1 можно считать закрытым, если notebook сверху вниз делает следующее:

1. принимает демо-тему CLTV;
2. строит research plan;
3. загружает seed-источники;
4. показывает coverage по research blocks;
5. скачивает или подхватывает cached raw/clean documents;
6. строит chunks;
7. фильтрует шум;
8. ранжирует chunks через BM25;
9. строит evidence table;
10. генерирует первый Markdown-отчет;
11. запускает quality gate;
12. явно показывает ограничения.

Оставшиеся ограничения не блокируют закрытие Этапа 1, но должны перейти в следующий этап:

- не все seed-источники скачались автоматически;
- HTML parser пока базовый;
- BM25 не понимает семантические совпадения;
- отчет пока template-based, без LLM synthesis;
- citation accuracy еще требует ручной оценки.